In [26]:
#### DATA LOADING ####

from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read Bronze table
df = spark.table("bronze.airport_traffic")

StatementMeta(, a2bd8ea6-aa4a-4b75-b8c9-c9150f8f8c4a, 31, Finished, Available, Finished, False)

In [27]:
#### DATA CLEANUP ####

# Rename columns for clarity
column_mapping = {
    "YEAR": "year",
    "MONTH_NUM": "month_number",
    "MONTH_MON": "month",
    "FLT_DATE": "flight_date",
    "APT_ICAO": "airport_code",
    "APT_NAME": "airport_name",
    "STATE_NAME": "state",
    "FLT_DEP_1": "total_departures",
    "FLT_ARR_1": "total_arrivals",
    "FLT_TOT_1": "total_flight_movements",
    "FLT_DEP_IFR_2": "ifr_departures",
    "FLT_ARR_IFR_2": "ifr_arrivals",
    "FLT_TOT_IFR_2": "ifr_flight_movements",
}

for old_name, new_name in column_mapping.items():
    df = df.withColumnRenamed(old_name, new_name)

# Clean and standardize data
df = (
    df
    .withColumn("airport_code", F.upper(F.trim(F.col("airport_code"))))
    .withColumn("airport_name", F.initcap(F.trim(F.col("airport_name"))))
    .withColumn("state", F.initcap(F.trim(F.col("state"))))
    .withColumn("month", F.upper(F.trim(F.col("month"))))
)

# Convert to date
df = df.withColumn(
    "flight_date",
    F.coalesce(
        F.to_date(F.col("flight_date"), "yyyy-MM-dd"),
        F.to_date(F.col("flight_date"), "dd/MM/yyyy"),
        F.to_date(F.col("flight_date"), "dd-MM-yyyy"),
        F.to_date(F.col("flight_date"), "yyyyMMdd"),
    )
)

# Convert to numeric
numeric_columns = [
    "year",
    "month_number",
    "total_departures",
    "total_arrivals",
    "total_flight_movements",
    "ifr_departures",
    "ifr_arrivals",
    "ifr_flight_movements",
]

for column_name in numeric_columns:
    df = df.withColumn(
        column_name,
        F.expr(f"try_cast(`{column_name}` as bigint)")
    )

# Add year_month column
df = df.withColumn(
    "year_month",
    F.date_format(F.col("flight_date"), "yyyyMM").cast("int")
)

StatementMeta(, a2bd8ea6-aa4a-4b75-b8c9-c9150f8f8c4a, 32, Finished, Available, Finished, False)

In [29]:
#### VALIDATION ####

# Validate airport codes
df = df.withColumn(
    "is_valid_airport_code",
    F.col("airport_code").rlike(r"^[A-Z]{4}$")
)

invalid_airport = df.filter(~F.col("is_valid_airport_code")).count()

if invalid_airport > 0:
    print(f"WARNING: {invalid_airport} rows have an invalid airport code.")
else:
    print("All airport codes are valid.")


# Validate flight dates
df = df.withColumn(
    "is_valid_date",
    F.col("flight_date").isNotNull()
)

invalid_date = df.filter(~F.col("is_valid_date")).count()

if invalid_date > 0:
    print(f"WARNING: {invalid_date} rows have an invalid flight date.")
else:
    print("All flight dates are valid.")


# Identify duplicate airport/date groups
duplicate_window = Window.partitionBy(
    "airport_code",
    "flight_date"
)

df = (
    df
    .withColumn(
        "_duplicate_count",
        F.count(F.lit(1)).over(duplicate_window)
    )
    .withColumn(
        "is_duplicate",
        F.col("_duplicate_count") > 1
    )
)

duplicate_count = df.filter(F.col("is_duplicate")).count()

if duplicate_count > 0:
    print(
        f"WARNING: {duplicate_count} rows belong to duplicate "
        "airport/date records."
    )
else:
    print("No duplicate airport/date records found.")


# Validate that all movement counts are non-negative
movement_columns = [
    "total_departures",
    "total_arrivals",
    "total_flight_movements",
]

ifr_columns = [
    "ifr_departures",
    "ifr_arrivals",
    "ifr_flight_movements",
]

valid_movements = reduce(
    lambda left, right: left & right,
    [
        F.col(column_name).isNotNull()
        & (F.col(column_name) >= 0)
        for column_name in movement_columns
    ]
)

valid_ifr = reduce(
    lambda left, right: left & right,
    [
        F.col(column_name).isNull()
        | (F.col(column_name) >= 0)
        for column_name in ifr_columns
    ]
)

df = df.withColumn(
    "is_valid_movements",
    valid_movements & valid_ifr
)

invalid_movements = df.filter(
    ~F.col("is_valid_movements")
).count()

if invalid_movements > 0:
    print(
        f"WARNING: {invalid_movements} rows have invalid movement counts."
    )
else:
    print("All movement counts are valid.")

StatementMeta(, a2bd8ea6-aa4a-4b75-b8c9-c9150f8f8c4a, 34, Finished, Available, Finished, False)

All airport codes are valid.
All flight dates are valid.
All movement counts are valid.


In [30]:
#### ACCEPTANCE CRITERIA ####

# A duplicate is not rejected. If it is an exact duplicate, one of the rows will be kept
df = df.withColumn(
    "is_valid_record",
    (
        F.col("is_valid_airport_code")
        & F.col("is_valid_date")
        & F.col("is_valid_movements")
    )
)

# Separate accepted and rejected records
accepted_df = df.filter(F.col("is_valid_record"))
rejected_df = df.filter(~F.col("is_valid_record"))

# Remove exact duplicate rows
accepted_df = accepted_df.dropDuplicates()

# Remove technical validation columns from accepted records
accepted_df = accepted_df.drop(
    "_duplicate_count",
    "is_valid_airport_code",
    "is_valid_date",
    "is_duplicate",
    "is_valid_movements",
    "is_valid_record",
)

# Keep validation flags in rejected records for investigation
rejected_df = rejected_df.drop("_duplicate_count")

accepted_count = accepted_df.count()
rejected_count = rejected_df.count()

print(f"Accepted airport traffic records: {accepted_count}")
print(f"Rejected airport traffic records: {rejected_count}")

StatementMeta(, a2bd8ea6-aa4a-4b75-b8c9-c9150f8f8c4a, 35, Finished, Available, Finished, False)

Accepted airport traffic records: 677921
Rejected airport traffic records: 0


In [31]:
#### WRITE TO SILVER ####

# Write accepted records to Silver
(
    accepted_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.airport_traffic")
)

# Write rejected records to Silver
(
    rejected_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.airport_traffic_rejected")
)

print("Silver tables written successfully.")

StatementMeta(, a2bd8ea6-aa4a-4b75-b8c9-c9150f8f8c4a, 36, Finished, Available, Finished, False)

Silver tables written successfully.


In [32]:
duplicate_rows = (
    df.filter(F.col("is_duplicate"))
      .orderBy("airport_code", "flight_date")
)

display(duplicate_rows)

StatementMeta(, a2bd8ea6-aa4a-4b75-b8c9-c9150f8f8c4a, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8cd78936-e149-4af6-8b86-a21118f8135e)

In [33]:
%%sql

-- 1. Row count
SELECT COUNT(*) AS row_count
FROM silver.airport_traffic;

StatementMeta(, a2bd8ea6-aa4a-4b75-b8c9-c9150f8f8c4a, 38, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [34]:
%%sql

-- 2. Date range and number of airports
SELECT
    MIN(flight_date) AS first_date,
    MAX(flight_date) AS last_date,
    COUNT(DISTINCT airport_code) AS airport_count
FROM silver.airport_traffic;

StatementMeta(, a2bd8ea6-aa4a-4b75-b8c9-c9150f8f8c4a, 39, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [35]:
%%sql

-- 12. Summary by year
SELECT
    year,
    COUNT(*) AS row_count,
    COUNT(DISTINCT airport_code) AS airport_count,
    SUM(total_departures) AS total_departures,
    SUM(total_arrivals) AS total_arrivals,
    SUM(total_flight_movements) AS total_movements
FROM silver.airport_traffic
GROUP BY year
ORDER BY year;

StatementMeta(, a2bd8ea6-aa4a-4b75-b8c9-c9150f8f8c4a, 40, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 6 fields>